# 02 — Robustez espacial e figuras

Reproduz o MQO com erros agrupados, o índice global de Moran, os modelos de erro espacial e as figuras derivadas da base analítica.

In [ ]:
from pathlib import Path
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.collections import PolyCollection
from scipy import sparse
from scipy.optimize import minimize_scalar
from scipy.sparse.csgraph import connected_components
from scipy.sparse.linalg import splu
from scipy.stats import norm

RAIZ_REPOSITORIO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_DADOS = RAIZ_REPOSITORIO / "data"
PASTA_RESULTADOS = RAIZ_REPOSITORIO / "outputs"
PASTA_RESULTADOS.mkdir(exist_ok=True)

ORDEM_HIERARQUIAS = ["H1", "H2", "H3", "H4"]
NOME_HIERARQUIA = {
    "H1": "Trânsito rápido",
    "H2": "Arterial",
    "H3": "Coletora",
    "H4": "Local",
}
CORES_HIERARQUIA = {
    "H1": "#315B7D",
    "H2": "#C84F3D",
    "H3": "#D3942A",
    "H4": "#3C8C84",
}

RENOMEAR_COLUNAS = {
    "H3_R11": "celula_h3_r11",
    "media_h3_kmh": "velocidade_operacional_media_kmh",
    "limite_contextual_kmh": "velocidade_contextual_referencia_kmh",
    "delta_v_kmh": "velocidade_insegura_kmh",
    "controle_cat": "tipo_controle",
    "intersecao": "intersecao_viaria",
    "densidade_classe": "densidade_domiciliar_classe",
    "comercio_relativo": "atividade_economica_relativa",
}
RENOMEAR_CONTROLES = {
    "Nenhum": "Nenhum",
    "Radar apenas": "Fiscalização eletrônica",
    "Semaforo apenas": "Controle semafórico",
    "Radar + semaforo": "Fiscalização e semáforo",
}

dados_h3 = pd.read_csv(
    PASTA_DADOS / "base_h3_analitica.csv.gz", dtype={"H3_R11": str}
).rename(columns=RENOMEAR_COLUNAS)
dados_h3["tipo_controle"] = dados_h3["tipo_controle"].replace(RENOMEAR_CONTROLES)
dados_h3 = dados_h3.sort_values(["hierarquia", "ordem_espacial"]).reset_index(drop=True)

arestas_vizinhanca = pd.read_csv(PASTA_DADOS / "vizinhanca_rede.csv.gz")

arquivo_residuos_rlm = PASTA_RESULTADOS / "residuos_RLM.csv.gz"
arquivo_coeficientes_rlm = PASTA_RESULTADOS / "coeficientes_RLM.csv"
if not arquivo_residuos_rlm.exists() or not arquivo_coeficientes_rlm.exists():
    raise FileNotFoundError("Execute primeiro o notebook 01_modelos_principais.ipynb")

residuos_rlm = pd.read_csv(arquivo_residuos_rlm, dtype={"celula_h3_r11": str})
coeficientes_rlm = pd.read_csv(arquivo_coeficientes_rlm)

## 1. Matrizes de vizinhança

In [ ]:
def construir_matriz_vizinhanca_binaria(hierarquia, distancia_m):
    n_celulas = int((dados_h3["hierarquia"] == hierarquia).sum())
    arestas_hierarquia = arestas_vizinhanca.loc[
        (arestas_vizinhanca["hierarquia"] == hierarquia)
        & (arestas_vizinhanca["distancia_m"] == distancia_m)
    ]

    origem = arestas_hierarquia["i"].to_numpy(int)
    destino = arestas_hierarquia["j"].to_numpy(int)
    linhas = np.r_[origem, destino]
    colunas = np.r_[destino, origem]

    return sparse.csr_matrix(
        (np.ones(len(linhas)), (linhas, colunas)),
        shape=(n_celulas, n_celulas),
    )


def padronizar_matriz_por_linha(matriz_vizinhanca):
    soma_linhas = np.asarray(matriz_vizinhanca.sum(axis=1)).ravel().astype(float)
    inverso_soma = np.zeros_like(soma_linhas)
    inverso_soma[soma_linhas > 0] = 1 / soma_linhas[soma_linhas > 0]
    return (
        sparse.diags(inverso_soma) @ matriz_vizinhanca.astype(float)
    ).tocsr()


registros_resumo_vizinhanca = []
for hierarquia in ORDEM_HIERARQUIAS:
    for distancia_m in [90, 150]:
        matriz_vizinhanca = construir_matriz_vizinhanca_binaria(
            hierarquia, distancia_m
        )
        n_componentes, _ = connected_components(matriz_vizinhanca, directed=False)
        registros_resumo_vizinhanca.append({
            "hierarquia": hierarquia,
            "distancia_m": distancia_m,
            "n": matriz_vizinhanca.shape[0],
            "arestas": matriz_vizinhanca.nnz // 2,
            "componentes_conectados": n_componentes,
        })

resumo_vizinhanca = pd.DataFrame(registros_resumo_vizinhanca)
print(resumo_vizinhanca.to_string(index=False))

## 2. MQO com erros agrupados pelos componentes de 150 m

In [ ]:
def construir_matriz_explicativa(dados_hierarquia, especificacao="completa"):
    matriz_explicativa = pd.DataFrame(index=dados_hierarquia.index)
    matriz_explicativa["Intercepto"] = 1.0

    if especificacao in {"completa", "infraestrutura"}:
        matriz_explicativa["Fiscalização eletrônica"] = (
            dados_hierarquia["tipo_controle"] == "Fiscalização eletrônica"
        ).astype(float)
        matriz_explicativa["Controle semafórico"] = (
            dados_hierarquia["tipo_controle"] == "Controle semafórico"
        ).astype(float)
        if (dados_hierarquia["tipo_controle"] == "Fiscalização e semáforo").any():
            matriz_explicativa["Fiscalização e semáforo"] = (
                dados_hierarquia["tipo_controle"] == "Fiscalização e semáforo"
            ).astype(float)
        matriz_explicativa["Interseção viária"] = dados_hierarquia[
            "intersecao_viaria"
        ].astype(float)

    if especificacao in {"completa", "contexto"}:
        matriz_explicativa["Densidade domiciliar (média)"] = (
            dados_hierarquia["densidade_domiciliar_classe"] == "Media"
        ).astype(float)
        matriz_explicativa["Densidade domiciliar (alta)"] = (
            dados_hierarquia["densidade_domiciliar_classe"] == "Alta"
        ).astype(float)
        matriz_explicativa["Atividade econômica relativa"] = dados_hierarquia[
            "atividade_economica_relativa"
        ].astype(float)

    return matriz_explicativa


registros_mqo_agrupado = []
for hierarquia in ORDEM_HIERARQUIAS:
    dados_hierarquia = dados_h3.loc[dados_h3["hierarquia"] == hierarquia].copy()
    matriz_vizinhanca_150m = construir_matriz_vizinhanca_binaria(hierarquia, 150)
    n_componentes, rotulo_componente = connected_components(
        matriz_vizinhanca_150m, directed=False
    )

    for nome_resposta, coluna_resposta in [
        ("velocidade_operacional_media", "velocidade_operacional_media_kmh"),
        ("velocidade_insegura", "velocidade_insegura_kmh"),
    ]:
        matriz_explicativa = construir_matriz_explicativa(
            dados_hierarquia, "completa"
        )
        modelo_mqo_agrupado = sm.OLS(
            dados_hierarquia[coluna_resposta].astype(float),
            matriz_explicativa,
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": rotulo_componente, "use_correction": True},
        )

        for variavel_explicativa in matriz_explicativa.columns:
            coeficiente = float(modelo_mqo_agrupado.params[variavel_explicativa])
            erro_padrao = float(modelo_mqo_agrupado.bse[variavel_explicativa])
            registros_mqo_agrupado.append({
                "hierarquia": hierarquia,
                "resposta": nome_resposta,
                "variavel": variavel_explicativa,
                "beta": coeficiente,
                "ep_agrupado": erro_padrao,
                "p_agrupado": float(
                    modelo_mqo_agrupado.pvalues[variavel_explicativa]
                ),
                "ic95_inf": coeficiente - 1.96 * erro_padrao,
                "ic95_sup": coeficiente + 1.96 * erro_padrao,
                "n_componentes": n_componentes,
            })

resultados_mqo_agrupado = pd.DataFrame(registros_mqo_agrupado)
resultados_mqo_agrupado.to_csv(
    PASTA_RESULTADOS / "MQO_erros_agrupados_150m.csv", index=False
)
print(resultados_mqo_agrupado.round(4).to_string(index=False))

## 3. Índice global de Moran

In [ ]:
def calcular_moran_global(
    valores, matriz_vizinhanca_binaria, gerador_aleatorio, n_permutacoes=999
):
    valores_centrados = np.asarray(valores, dtype=float)
    valores_centrados = valores_centrados - valores_centrados.mean()
    matriz_pesos = padronizar_matriz_por_linha(matriz_vizinhanca_binaria)

    n_observacoes = len(valores_centrados)
    soma_pesos = float(matriz_pesos.sum())
    moran_observado = (
        (n_observacoes / soma_pesos)
        * float(valores_centrados @ (matriz_pesos @ valores_centrados))
        / float(valores_centrados @ valores_centrados)
    )
    moran_esperado = -1 / (n_observacoes - 1)

    moran_permutado = np.empty(n_permutacoes)
    for indice_permutacao in range(n_permutacoes):
        valores_permutados = gerador_aleatorio.permutation(valores_centrados)
        moran_permutado[indice_permutacao] = (
            (n_observacoes / soma_pesos)
            * float(valores_permutados @ (matriz_pesos @ valores_permutados))
            / float(valores_permutados @ valores_permutados)
        )

    p_permutacao = (
        1
        + np.sum(
            np.abs(moran_permutado - moran_esperado)
            >= abs(moran_observado - moran_esperado)
        )
    ) / (n_permutacoes + 1)

    return moran_observado, p_permutacao, moran_esperado


registros_moran = []
for indice_hierarquia, hierarquia in enumerate(ORDEM_HIERARQUIAS, 1):
    dados_hierarquia = dados_h3.loc[dados_h3["hierarquia"] == hierarquia].copy()
    dados_hierarquia = (
        dados_hierarquia.merge(
            residuos_rlm.loc[residuos_rlm["hierarquia"] == hierarquia],
            on=["hierarquia", "celula_h3_r11"],
            validate="one_to_one",
        )
        .sort_values("ordem_espacial")
    )

    componentes_analisados = {
        "velocidade_operacional_media": dados_hierarquia[
            "velocidade_operacional_media_kmh"
        ],
        "velocidade_operacional_media_predita_rlm": dados_hierarquia[
            "velocidade_operacional_media_predita_kmh"
        ],
        "residuo_rlm": dados_hierarquia[
            "residuo_velocidade_operacional_media_kmh"
        ],
        "velocidade_contextual_referencia": dados_hierarquia[
            "velocidade_contextual_referencia_kmh"
        ],
        "velocidade_insegura": dados_hierarquia["velocidade_insegura_kmh"],
    }

    for distancia_m in [90, 150]:
        matriz_vizinhanca = construir_matriz_vizinhanca_binaria(
            hierarquia, distancia_m
        )
        gerador_aleatorio = np.random.default_rng(
            20260828 + 10000 * indice_hierarquia + distancia_m
        )

        for componente, valores in componentes_analisados.items():
            moran_i, p_permutacao, moran_esperado = calcular_moran_global(
                valores, matriz_vizinhanca, gerador_aleatorio, n_permutacoes=999
            )
            registros_moran.append({
                "hierarquia": hierarquia,
                "distancia_m": distancia_m,
                "componente": componente,
                "moran_I": moran_i,
                "p_permutacao": p_permutacao,
                "esperado": moran_esperado,
            })

resultados_moran = pd.DataFrame(registros_moran)
resultados_moran.to_csv(PASTA_RESULTADOS / "Moran_90_150m.csv", index=False)
print(
    resultados_moran.query("distancia_m == 150").round(4).to_string(index=False)
)

## 4. Modelo de erro espacial em 90 e 150 m

In [ ]:
def calcular_moran_sem_permutacao(valores, matriz_pesos):
    valores_centrados = np.asarray(valores, dtype=float)
    valores_centrados = valores_centrados - valores_centrados.mean()
    n_observacoes = len(valores_centrados)
    return (
        (n_observacoes / float(matriz_pesos.sum()))
        * float(valores_centrados @ (matriz_pesos @ valores_centrados))
        / float(valores_centrados @ valores_centrados)
    )


def ajustar_modelo_erro_espacial(
    variavel_resposta, matriz_explicativa, matriz_pesos
):
    variavel_resposta = np.asarray(variavel_resposta, dtype=float)
    matriz_explicativa = np.asarray(matriz_explicativa, dtype=float)
    n_observacoes = len(variavel_resposta)

    resposta_vizinhos = matriz_pesos @ variavel_resposta
    explicativas_vizinhos = matriz_pesos @ matriz_explicativa
    matriz_identidade = sparse.eye(n_observacoes, format="csc")

    def funcao_objetivo(lambda_espacial, retornar_detalhes=False):
        resposta_transformada = (
            variavel_resposta - lambda_espacial * resposta_vizinhos
        )
        explicativas_transformadas = (
            matriz_explicativa - lambda_espacial * explicativas_vizinhos
        )

        produto_xx = explicativas_transformadas.T @ explicativas_transformadas
        produto_xy = explicativas_transformadas.T @ resposta_transformada
        try:
            coeficientes = np.linalg.solve(produto_xx, produto_xy)
        except np.linalg.LinAlgError:
            coeficientes = np.linalg.lstsq(
                produto_xx, produto_xy, rcond=None
            )[0]

        inovacao = resposta_transformada - explicativas_transformadas @ coeficientes
        variancia = float(inovacao @ inovacao) / n_observacoes
        decomposicao_lu = splu(
            (matriz_identidade - lambda_espacial * matriz_pesos).tocsc()
        )
        log_determinante = float(
            np.log(np.abs(decomposicao_lu.U.diagonal())).sum()
        )
        log_verossimilhanca = (
            log_determinante
            - 0.5
            * n_observacoes
            * (np.log(2 * np.pi) + 1 + np.log(variancia))
        )

        if retornar_detalhes:
            return (
                log_verossimilhanca,
                coeficientes,
                inovacao,
                variancia,
                produto_xx,
            )
        return -log_verossimilhanca

    otimizacao = minimize_scalar(
        funcao_objetivo,
        bounds=(-0.98, 0.98),
        method="bounded",
        options={"xatol": 1e-7, "maxiter": 80},
    )

    lambda_espacial = float(otimizacao.x)
    (
        log_verossimilhanca,
        coeficientes,
        inovacao,
        variancia,
        produto_xx,
    ) = funcao_objetivo(lambda_espacial, retornar_detalhes=True)

    matriz_covariancia = variancia * np.linalg.inv(produto_xx)
    erros_padrao = np.sqrt(np.diag(matriz_covariancia))
    p_valores = 2 * norm.sf(np.abs(coeficientes / erros_padrao))
    residuo_bruto = variavel_resposta - matriz_explicativa @ coeficientes

    passo = 1e-4
    valor_central = -funcao_objetivo(lambda_espacial)
    valor_esquerda = -funcao_objetivo(max(-0.9799, lambda_espacial - passo))
    valor_direita = -funcao_objetivo(min(0.9799, lambda_espacial + passo))
    segunda_derivada = (
        valor_direita - 2 * valor_central + valor_esquerda
    ) / passo**2
    erro_padrao_lambda = (
        np.sqrt(-1 / segunda_derivada) if segunda_derivada < 0 else np.nan
    )

    return {
        "lambda": lambda_espacial,
        "ep_lambda": erro_padrao_lambda,
        "loglik": log_verossimilhanca,
        "beta": coeficientes,
        "ep": erros_padrao,
        "p": p_valores,
        "sigma2": variancia,
        "residuo_bruto": residuo_bruto,
        "inovacao": inovacao,
        "sucesso": bool(otimizacao.success),
    }


registros_coeficientes_sem = []
registros_diagnosticos_sem = []

for distancia_m in [90, 150]:
    for hierarquia in ORDEM_HIERARQUIAS:
        dados_hierarquia = dados_h3.loc[dados_h3["hierarquia"] == hierarquia].copy()
        matriz_pesos = padronizar_matriz_por_linha(
            construir_matriz_vizinhanca_binaria(hierarquia, distancia_m)
        )

        for nome_resposta, coluna_resposta, especificacao in [
            ("velocidade_operacional_media", "velocidade_operacional_media_kmh", "completa"),
            ("velocidade_insegura", "velocidade_insegura_kmh", "completa"),
            ("velocidade_insegura_infraestrutura", "velocidade_insegura_kmh", "infraestrutura"),
        ]:
            matriz_explicativa = construir_matriz_explicativa(
                dados_hierarquia, especificacao
            )
            resultado_sem = ajustar_modelo_erro_espacial(
                dados_hierarquia[coluna_resposta],
                matriz_explicativa,
                matriz_pesos,
            )

            for indice_variavel, variavel_explicativa in enumerate(
                matriz_explicativa.columns
            ):
                registros_coeficientes_sem.append({
                    "distancia_m": distancia_m,
                    "hierarquia": hierarquia,
                    "resposta": nome_resposta,
                    "variavel": variavel_explicativa,
                    "beta": resultado_sem["beta"][indice_variavel],
                    "ep_condicional_lambda": resultado_sem["ep"][indice_variavel],
                    "p_condicional_lambda": resultado_sem["p"][indice_variavel],
                })

            n_parametros = matriz_explicativa.shape[1] + 2
            registros_diagnosticos_sem.append({
                "distancia_m": distancia_m,
                "hierarquia": hierarquia,
                "resposta": nome_resposta,
                "n": len(dados_hierarquia),
                "lambda": resultado_sem["lambda"],
                "ep_lambda_aprox": resultado_sem["ep_lambda"],
                "AIC": -2 * resultado_sem["loglik"] + 2 * n_parametros,
                "moran_residuo_bruto": calcular_moran_sem_permutacao(
                    resultado_sem["residuo_bruto"], matriz_pesos
                ),
                "moran_inovacao": calcular_moran_sem_permutacao(
                    resultado_sem["inovacao"], matriz_pesos
                ),
                "sucesso": resultado_sem["sucesso"],
            })

coeficientes_sem = pd.DataFrame(registros_coeficientes_sem)
diagnosticos_sem = pd.DataFrame(registros_diagnosticos_sem)

coeficientes_sem.to_csv(
    PASTA_RESULTADOS / "SEM_coeficientes_90_150m.csv", index=False
)
diagnosticos_sem.to_csv(
    PASTA_RESULTADOS / "SEM_diagnosticos_90_150m.csv", index=False
)
print(diagnosticos_sem.round(4).to_string(index=False))

## 5. Figuras

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9})

figura_h3, eixos_h3 = plt.subplots(2, 2, figsize=(7.2, 7.2))
for eixo, hierarquia, letra in zip(
    eixos_h3.flat, ORDEM_HIERARQUIAS, ["(a)", "(b)", "(c)", "(d)"]
):
    celulas_h3 = dados_h3.loc[
        dados_h3["hierarquia"] == hierarquia, "celula_h3_r11"
    ]
    poligonos_h3 = [
        [(longitude, latitude) for latitude, longitude in h3.cell_to_boundary(celula)]
        for celula in celulas_h3
    ]
    colecao_h3 = PolyCollection(
        poligonos_h3,
        facecolor=CORES_HIERARQUIA[hierarquia],
        edgecolor="none",
        alpha=0.92,
    )
    eixo.add_collection(colecao_h3)
    eixo.autoscale_view()
    eixo.set_aspect(1 / np.cos(np.deg2rad(-25.45)))
    eixo.text(
        0.02, 0.98, letra,
        transform=eixo.transAxes,
        ha="left", va="top", fontweight="bold",
    )
    eixo.axis("off")

figura_h3.tight_layout(pad=0.5)
figura_h3.savefig(
    PASTA_RESULTADOS / "figura_4_1_celulas_H3_hierarquia.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

variaveis_explicativas = [
    "Fiscalização eletrônica",
    "Controle semafórico",
    "Fiscalização e semáforo",
    "Interseção viária",
    "Densidade domiciliar (média)",
    "Densidade domiciliar (alta)",
    "Atividade econômica relativa",
]

dados_coeficientes_figura = coeficientes_rlm.query(
    "resposta == 'velocidade_operacional_media' and variavel in @variaveis_explicativas"
)

figura_coeficientes, eixo_coeficientes = plt.subplots(figsize=(8.0, 5.0))
posicoes_variaveis = np.arange(len(variaveis_explicativas))

for deslocamento, hierarquia in zip(
    [-0.27, -0.09, 0.09, 0.27], ORDEM_HIERARQUIAS
):
    coeficientes_hierarquia = (
        dados_coeficientes_figura.loc[
            dados_coeficientes_figura["hierarquia"] == hierarquia
        ]
        .set_index("variavel")
    )
    variaveis_presentes = [
        variavel
        for variavel in variaveis_explicativas
        if variavel in coeficientes_hierarquia.index
    ]
    posicoes_hierarquia = np.array([
        variaveis_explicativas.index(variavel)
        for variavel in variaveis_presentes
    ]) + deslocamento
    valores_hierarquia = coeficientes_hierarquia.loc[variaveis_presentes]

    eixo_coeficientes.errorbar(
        valores_hierarquia["beta"],
        posicoes_hierarquia,
        xerr=[
            valores_hierarquia["beta"] - valores_hierarquia["ic95_inf"],
            valores_hierarquia["ic95_sup"] - valores_hierarquia["beta"],
        ],
        fmt="o",
        capsize=2,
        color=CORES_HIERARQUIA[hierarquia],
        label=NOME_HIERARQUIA[hierarquia],
    )

eixo_coeficientes.axvline(0, color="#6C757D", linewidth=0.8)
eixo_coeficientes.set_yticks(posicoes_variaveis, variaveis_explicativas)
eixo_coeficientes.invert_yaxis()
eixo_coeficientes.set_xlabel("Coeficiente (km/h)")
eixo_coeficientes.grid(axis="x", color="#E4E7E9", linewidth=0.6)
eixo_coeficientes.legend(frameon=False, ncol=2)

figura_coeficientes.tight_layout()
figura_coeficientes.savefig(
    PASTA_RESULTADOS / "coeficientes_RLM_velocidade_media.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Saídas

Os resultados e figuras são gravados em `outputs/`.